- Loads processed IS-volume output files generated by the previous batch extraction script.
- Extracts metadata from the file names, including date, run, condition, measurement ID, and IS-threshold class.
- Reads the already-normalized volume values from each file; no additional normalization is performed in this script.
- Uses the real elapsed time column (`Time [min]` or `Time [s]`) for plotting; seconds are converted to minutes.
- If no real elapsed-time column is present, time is estimated from the numeric time-point index using the fallback `TIME_STEP_MIN`.
- Groups data by threshold, condition, and time point.
- Calculates the mean normalized volume, standard deviation, and number of measurements for each condition/time point.
- Saves summary tables in long and wide CSV format.
- Generates plots of mean normalized IS volume over time with SD shading for each condition.

In [ ]:
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# =========================
# SETTINGS
# =========================
INPUT_DIR = Path(r" ***** \SampleData\Individual-run_TestData_Script1-3\run1_ImarisSurfaces\Time_offset_corrected\batch_IS_outputs")   # <-- change this
OUTPUT_DIR = INPUT_DIR / "processed_plots"
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_THRESHOLDS = ["250", "500", "auto"]
CONDITION_ORDER = ["Nivo", "Durva", "mw11h317", "Enva", "mock", "aPD", "aPDL"]

# --- Fallback time conversion settings ---
# The script now uses the real elapsed time from a "Time [min]" column for plotting.
# These values are only used for older files that do not contain a usable "Time [min]" column.
TIME_STEP_MIN = 5
FIRST_TIMEPOINT_VALUE = 1


In [ ]:
# =========================
# HELPERS
# =========================
def normalize_text(s):
    """Lowercase and remove non-alphanumeric characters."""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def find_required_columns(df):
    """
    Find required columns robustly, even if spacing/capitalization varies.

    Intended input format:
        Time              -> numeric time-point used for grouping
        Time [min]        -> real elapsed time used for plotting/output x-axis in minutes
        Time [s]          -> real elapsed time used for plotting/output x-axis, converted to minutes
        Normalized Volume -> y-value

    Grouping logic stays unchanged: the plain numeric Time column is the time-point.
    Plotting uses real elapsed time in minutes. If neither Time [min] nor Time [s]
    exists, the script falls back to the old TIME_STEP_MIN logic.
    """
    col_map = {col: normalize_text(col) for col in df.columns}

    timepoint_col = None
    elapsed_time_col = None
    elapsed_time_unit = None  # "min" or "s"
    normvol_col = None

    # Prefer the plain "Time" column as the grouping time-point.
    # This avoids accidentally using "Time [min]" or "Time [s]" as the grouping variable.
    for original, cleaned in col_map.items():
        if cleaned in {"time", "timepoint", "timepointindex", "timeindex"}:
            timepoint_col = original
            break

    minute_names = {
        "timemin", "timeinmin", "timeminutes", "timeinminutes",
        "elapsedtimemin", "elapsedtimeminutes"
    }
    second_names = {
        "times", "timeins", "timesecond", "timeseconds", "timeinseconds",
        "elapsedtimes", "elapsedtimesecond", "elapsedtimeseconds"
    }

    # Prefer minute columns if present, because those can be used directly.
    for original, cleaned in col_map.items():
        if original == timepoint_col:
            continue
        if cleaned in minute_names or (("time" in cleaned) and ("min" in cleaned)):
            elapsed_time_col = original
            elapsed_time_unit = "min"
            break

    # Otherwise accept seconds and convert them to minutes later.
    if elapsed_time_col is None:
        for original, cleaned in col_map.items():
            if original == timepoint_col:
                continue
            if cleaned in second_names or (("time" in cleaned) and ("sec" in cleaned)):
                elapsed_time_col = original
                elapsed_time_unit = "s"
                break

    # Fallback if there is no exact plain time column.
    # Use a time-like column that is not the real elapsed-time column.
    if timepoint_col is None:
        for original, cleaned in col_map.items():
            if cleaned.startswith("time") and original != elapsed_time_col:
                timepoint_col = original
                break

    for original, cleaned in col_map.items():
        if "normalized" in cleaned and "volume" in cleaned:
            normvol_col = original
            break

    if timepoint_col is None:
        raise ValueError(f"Could not find a grouping time-point column such as 'Time' in columns: {list(df.columns)}")
    if normvol_col is None:
        raise ValueError(f"Could not find a 'Normalized Volume' column in columns: {list(df.columns)}")

    return timepoint_col, elapsed_time_col, elapsed_time_unit, normvol_col


# Canonical condition names used for grouping/plotting.
# Note: aPDL is listed before aPD so that aPDL is not accidentally shortened to aPD.
CONDITION_PATTERN = r"aPDL|aPD|mw11h317|Nivo|Durva|Enva|mock"
CONDITION_CANONICAL = {
    "nivo": "Nivo",
    "durva": "Durva",
    "enva": "Enva",
    "mock": "mock",
    "mw11h317": "mw11h317",
    "apd": "aPD",
    "apdl": "aPDL",
}


def parse_filename(file_path):
    """
    Parse the measurement metadata from the file name.

    New expected input names:
        XXXXXX_runX_(condition)_X-X.csv
        XXXXXX_runX_5ug_(condition)_X-X.csv

    The function also accepts batch-output names that still contain an IS threshold:
        XXXXXX_runX_(condition)_X-X_IS-250.csv
        XXXXXX_runX_5ug_(condition)_X-X_IS-auto.csv

    For backward compatibility it also accepts the older no-underscore form:
        XXXXXX_runX_Nivo1-1_IS-250.csv
    """
    stem = file_path.stem

    pattern = re.compile(
        rf"^"
        rf"(?P<date>\d{{6}})_"
        rf"run(?P<run>\d+)"
        rf"(?:_(?P<extra>5ug))?"
        rf"_(?P<condition_raw>{CONDITION_PATTERN})"
        rf"_?(?P<file_number>\d+-\d+)"
        rf"(?:_IS-(?P<threshold>250|500|auto))?"
        rf"$",
        re.IGNORECASE,
    )

    match = pattern.match(stem)
    if not match:
        return None

    condition_raw = match.group("condition_raw")
    condition_key = condition_raw.lower()
    condition = CONDITION_CANONICAL.get(condition_key)
    if condition is None:
        return None

    threshold = match.group("threshold")
    if threshold is None:
        # If files do not have an _IS-250/_IS-500/_IS-auto suffix, process them as one pooled class.
        threshold = "all"
    else:
        threshold = threshold.lower()

    return {
        "date": match.group("date"),
        "run": f"run{match.group('run')}",
        "extra": match.group("extra") or "",
        "condition_raw": condition_raw,
        "condition": condition,
        "measurement": match.group("file_number"),
        "threshold": threshold,
    }


def read_table_file(file_path):
    """
    Read csv/txt/tsv/xlsx/xls.
    """
    suffix = file_path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)
    else:
        # sep=None lets pandas infer comma/tab/semicolon, etc.
        df = pd.read_csv(file_path, sep=None, engine="python")

    return df


In [ ]:
# =========================
# LOAD ALL FILES
# =========================
all_records = []

supported_files = []
for ext in ["*.csv", "*.txt", "*.tsv", "*.xlsx", "*.xls"]:
    supported_files.extend(INPUT_DIR.glob(ext))

if not supported_files:
    raise FileNotFoundError(f"No supported files found in: {INPUT_DIR}")

for file_path in supported_files:
    info = parse_filename(file_path)
    if info is None:
        print(f"Skipping file with non-matching name: {file_path.name}")
        continue

    try:
        df = read_table_file(file_path)
        timepoint_col, elapsed_time_col, elapsed_time_unit, normvol_col = find_required_columns(df)

        if elapsed_time_col is not None:
            tmp = df[[timepoint_col, elapsed_time_col, normvol_col]].copy()
            tmp.columns = ["timepoint", "elapsed_time_raw", "normalized_volume"]
        else:
            tmp = df[[timepoint_col, normvol_col]].copy()
            tmp.columns = ["timepoint", "normalized_volume"]

        tmp["timepoint"] = pd.to_numeric(tmp["timepoint"], errors="coerce")
        tmp["normalized_volume"] = pd.to_numeric(tmp["normalized_volume"], errors="coerce")

        if elapsed_time_col is not None:
            # Preferred behavior: use the actual elapsed time from the file.
            # If the file uses seconds, convert to minutes so the x-axis is always in minutes.
            tmp["elapsed_time_raw"] = pd.to_numeric(tmp["elapsed_time_raw"], errors="coerce")

            if elapsed_time_unit == "min":
                tmp["time_min"] = tmp["elapsed_time_raw"]
            elif elapsed_time_unit == "s":
                tmp["time_min"] = tmp["elapsed_time_raw"] / 60.0
            else:
                raise ValueError(
                    f"Internal error: unknown elapsed-time unit '{elapsed_time_unit}' "
                    f"for column '{elapsed_time_col}' in {file_path.name}"
                )

            tmp["time_source_column"] = elapsed_time_col
            tmp["time_source_unit"] = elapsed_time_unit
            tmp = tmp.drop(columns=["elapsed_time_raw"])
        else:
            # Backward-compatible fallback for older files without a real Time [min] or Time [s] column.
            tmp["time_min"] = (tmp["timepoint"] - FIRST_TIMEPOINT_VALUE) * TIME_STEP_MIN
            tmp["time_source_column"] = "fallback_TIME_STEP_MIN"
            tmp["time_source_unit"] = "min"
            print(
                f"Warning: {file_path.name} has no 'Time [min]' or 'Time [s]' column. "
                f"Using fallback TIME_STEP_MIN={TIME_STEP_MIN}."
            )

        tmp = tmp.dropna(subset=["timepoint", "time_min", "normalized_volume"])

        tmp["date"] = info["date"]
        tmp["run"] = info["run"]
        tmp["extra"] = info["extra"]          # e.g. "5ug" if present, otherwise empty
        tmp["condition"] = info["condition"]  # canonical condition used for grouping
        tmp["condition_raw"] = info["condition_raw"]
        tmp["measurement"] = info["measurement"]
        tmp["threshold"] = info["threshold"]
        tmp["source_file"] = file_path.name

        all_records.append(tmp)

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")

if not all_records:
    raise ValueError("No valid data could be loaded from the files.")

data = pd.concat(all_records, ignore_index=True)

# Decide which threshold classes to process.
# If your file names still contain _IS-250/_IS-500/_IS-auto, the script behaves as before.
# If your file names no longer contain an _IS-... suffix, all files are processed under threshold == "all".
available_thresholds = set(data["threshold"].unique())

if available_thresholds == {"all"}:
    thresholds_to_process = ["all"]
else:
    thresholds_to_process = [t for t in TARGET_THRESHOLDS if t in available_thresholds]

if not thresholds_to_process:
    raise ValueError(
        "No files matched the requested thresholds. "
        f"Available threshold labels were: {sorted(available_thresholds)}"
    )

data = data[data["threshold"].isin(thresholds_to_process)].copy()

print("Loaded files:")
loaded_overview = (
    data[["source_file", "date", "run", "extra", "condition", "measurement", "threshold", "time_source_column", "time_source_unit"]]
    .drop_duplicates()
    .sort_values(["condition", "date", "run", "measurement", "threshold", "source_file"])
)
print(loaded_overview.to_string(index=False))


# =========================
# AVERAGE BY CONDITION + TIME-POINT + THRESHOLD
# =========================
# Important:
#   - timepoint is still the grouping variable
#   - time_min is the real elapsed time in minutes, used later for plotting/output
#   - Time [min] is used directly
#   - Time [s] is converted to minutes by dividing by 60
#   - if several files have slightly different real time values for the same timepoint,
#     the plotted x-position is the mean time_min for that timepoint/condition.
summary = (
    data
    .groupby(["threshold", "condition", "timepoint"], as_index=False)
    .agg(
        time_min=("time_min", "mean"),
        time_min_sd=("time_min", "std"),
        mean_normalized_volume=("normalized_volume", "mean"),
        sd_normalized_volume=("normalized_volume", "std"),
        n=("normalized_volume", "count")
    )
)

summary = summary.sort_values(["threshold", "condition", "timepoint"])


# =========================
# SAVE CSVs + PLOTS
# =========================
for threshold in thresholds_to_process:
    sub = summary[summary["threshold"] == threshold].copy()
    if sub.empty:
        print(f"No data found for threshold class: {threshold}")
        continue

    if threshold == "all":
        output_label = "all"
        plot_title = "Normalized Volume vs Time"
    else:
        output_label = f"IS-{threshold}"
        plot_title = f"Normalized Volume vs Time (IS-{threshold})"

    # -------- Long-format CSV --------
    long_csv = OUTPUT_DIR / f"summary_{output_label}_long.csv"
    sub.to_csv(long_csv, index=False)

    # -------- Wide-format CSV --------
    # Use timepoint as the pivot index so conditions stay aligned.
    # The wide table includes condition-specific time_min columns because different
    # conditions/files can now have different acquisition intervals.
    wide_parts = []
    for metric in ["time_min", "time_min_sd", "mean_normalized_volume", "sd_normalized_volume", "n"]:
        pivot = sub.pivot(index="timepoint", columns="condition", values=metric)
        pivot.columns = [f"{cond}_{metric}" for cond in pivot.columns]
        wide_parts.append(pivot)

    wide_df = pd.concat(wide_parts, axis=1).reset_index()
    wide_df = wide_df.sort_values("timepoint")

    # Add actual elapsed time in minutes. This is the x-axis time used for plotting.
    # Averaged across all available conditions for each numeric timepoint.
    time_lookup = (
        sub.groupby("timepoint", as_index=False)
        .agg(
            time_min=("time_min", "mean"),
            time_min_sd=("time_min", "std")
        )
        .sort_values("timepoint")
    )
    wide_df = pd.merge(time_lookup, wide_df, on="timepoint", how="left")

    ordered_cols = ["timepoint", "time_min", "time_min_sd"]
    for cond in CONDITION_ORDER:
        for metric in ["time_min", "time_min_sd", "mean_normalized_volume", "sd_normalized_volume", "n"]:
            col = f"{cond}_{metric}"
            if col in wide_df.columns:
                ordered_cols.append(col)

    remaining_cols = [c for c in wide_df.columns if c not in ordered_cols]
    wide_df = wide_df[ordered_cols + remaining_cols]

    wide_csv = OUTPUT_DIR / f"summary_{output_label}_wide.csv"
    wide_df.to_csv(wide_csv, index=False)

    # -------- Plot --------
    plt.figure(figsize=(8, 5))

    for condition in CONDITION_ORDER:
        cond_df = sub[sub["condition"] == condition].sort_values("timepoint")
        if cond_df.empty:
            continue

        # x-axis always uses real elapsed time in minutes.
        # "Time [min]" is used directly; "Time [s]" was converted above.
        x = cond_df["time_min"]
        y = cond_df["mean_normalized_volume"]
        sd = cond_df["sd_normalized_volume"].fillna(0)

        plt.plot(x, y, marker="o", label=condition)
        plt.fill_between(x, y - sd, y + sd, alpha=0.2)

    plt.xlabel("Time (min)")
    plt.ylabel("Normalized Volume")
    plt.title(plot_title)
    plt.legend()
    plt.tight_layout()

    plot_file = OUTPUT_DIR / f"plot_{output_label}.png"
    plt.savefig(plot_file, dpi=300)
    plt.close()

print(f"Done. Results saved in: {OUTPUT_DIR}")
